In [ ]:
# !git clone (git repo)
# %cd sudoku-residual
# !uv sync --extra tpu --quiet

In [ ]:
# copy the training traces to the root
!cp /content/drive/MyDrive/bt_traces_3m.npz ./


In [ ]:
%%writefile experiments_local.py

COMMON = dict(
    dtype="bfloat16",
    batch_size=512,
    num_tokens= 6 * 198_253_000,
    eval_every=500,
    num_checkpoints=9,
    n_layers=8,
    n_heads=8,
    d_model=576,
    d_ff=3456,
    loss_mask="after_clues",
    weight_decay=0.1,
    warmup_tokens=5_000_000,
    seed=42,
    pack_len=300,
)

EXPERIMENTS = [
    ("3M-backtracking-packing", {"lr": 1e-3, "schedule_type": "cosine", "traces_path": "bt_traces_3m.npz"})
]

In [ ]:
import datetime, os, subprocess, sys, zipfile
from pathlib import Path
from google.colab import drive, runtime

# ── Config ───────────────────────────────────────────────────────────────────
DRIVE_DIR  = '/content/drive/MyDrive/sudoku_results'
ARCHIVE_TAG = ''   # optional label, e.g. 'seed_sweep'

# ── 0. Mount Drive & create timestamped output folder ────────────────────────
# drive.mount('/content/drive')
# os.makedirs(DRIVE_DIR, exist_ok=True)

timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
tag = f'_{ARCHIVE_TAG}' if ARCHIVE_TAG else ''
run_dir = os.path.join(DRIVE_DIR, f'{timestamp}{tag}')
os.makedirs(run_dir, exist_ok=True)
print(f'Output folder: {run_dir}')

def run(cmd):
    """Stream stdout+stderr live into cell output, preserving \\r progress lines."""
    print(f'\n>>> {" ".join(cmd)}\n', flush=True)
    proc = subprocess.Popen(
        cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=0,
    )
    while chunk := proc.stdout.read(256):
        sys.stdout.write(chunk.decode('utf-8', errors='replace'))
        sys.stdout.flush()
    proc.wait()
    if proc.returncode != 0:
        raise subprocess.CalledProcessError(proc.returncode, cmd)

def experiment_names():
    return sorted(p.parent.name for p in Path('results').glob('*/config.json'))

# ── 1. Train ─────────────────────────────────────────────────────────────────
run(['uv', 'run', 'python', 'scripts/run_training.py'])

# ── 2. Collect traces for every checkpoint of every experiment ────────────────
for name in experiment_names():
    run(['uv', 'run', 'python', 'scripts/collect_activations.py',
         '--traces-only', '--all-steps', '--name', name])

# ── 3. Evaluate every checkpoint of every experiment ─────────────────────────
for name in experiment_names():
    run(['uv', 'run', 'python', 'scripts/run_eval.py', '--all-steps', '--name', name])

# ── 4. Archive results (traces + evals, no checkpoints) ──────────────────────
# ZIP_DEFLATED: text files and .npz compress well
results_zip = os.path.join(run_dir, 'results.zip')
print(f'\nArchiving results → {results_zip} ...')
with zipfile.ZipFile(results_zip, 'w', zipfile.ZIP_DEFLATED) as zf:
    for exp_dir in sorted(Path('results').iterdir()):
        if not exp_dir.is_dir():
            continue
        for fname in ['train_log.json', 'config.json']:
            fpath = exp_dir / fname
            if fpath.exists():
                zf.write(fpath, fpath); print(f'  + {fpath}')
        steps_dir = exp_dir / 'steps'
        if steps_dir.is_dir():
            for step_dir in sorted(steps_dir.iterdir(), key=lambda p: int(p.name)):
                for fname in ['activations.npz', 'eval.txt']:
                    fpath = step_dir / fname
                    if fpath.exists():
                        zf.write(fpath, fpath); print(f'  + {fpath}')
print(f'  {os.path.getsize(results_zip)/1e6:.1f} MB')

# ── 5. Archive checkpoints (one zip per experiment) ───────────────────────────
print(f'\nArchiving checkpoints ...')
for exp_dir in sorted(Path('results').iterdir()):
    if not exp_dir.is_dir():
        continue
    ckpt_dir = exp_dir / 'checkpoint'
    if not ckpt_dir.is_dir():
        continue
    ckpt_zip = os.path.join(run_dir, f'{exp_dir.name}_checkpoint.zip')
    with zipfile.ZipFile(ckpt_zip, 'w', zipfile.ZIP_STORED) as zf:
        for f in sorted(ckpt_dir.rglob('*')):
            if f.is_file():
                zf.write(f, f)
    print(f'  {exp_dir.name} → {os.path.getsize(ckpt_zip)/1e6:.1f} MB')

# ── 6. Terminate ──────────────────────────────────────────────────────────────
print(f'\nAll saved to {run_dir}\nTerminating runtime...')
runtime.unassign()